In [1]:
# %% [markdown]
# # 07 — Risk Scoring Target Design
#
# ## Objective
# Define the first target variables for the SecureHealth risk scoring module.
#
# This notebook will:
# - load the analytical member-level bases
# - review candidate risk variables
# - create initial target definitions
# - produce a first target-ready modeling dataset
#
# Main outputs:
# - `member_risk_model_base.csv`

# %%
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# %%
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR, OUTPUTS_DIR

# %% [markdown]
# ## Load processed data

# %%
member_master = pd.read_csv(PROCESSED_DIR / "member_master.csv")
claims_analytical_base = pd.read_csv(PROCESSED_DIR / "claims_analytical_base.csv")

print("member_master:", member_master.shape)
print("claims_analytical_base:", claims_analytical_base.shape)

# %%
display(member_master.head(3))
display(claims_analytical_base.head(3))

# %% [markdown]
# ## Detect claim cost column

# %%
claim_cost_col = None
for candidate in ["claim_amount", "paid_amount", "approved_amount", "amount_paid", "cost_amount", "total_cost"]:
    if candidate in claims_analytical_base.columns:
        claim_cost_col = candidate
        break

print("Detected claim cost column:", claim_cost_col)

# %% [markdown]
# ## Build member-level claims aggregates

# %%
member_claims_agg = None

if "member_id" in claims_analytical_base.columns:
    agg_parts = {"member_id": "size"}

    if claim_cost_col is not None:
        agg_df = (
            claims_analytical_base.groupby("member_id")[claim_cost_col]
            .agg(["count", "mean", "median", "sum", "max"])
            .reset_index()
        )
        agg_df.columns = [
            "member_id",
            "claims_count",
            "claim_cost_mean",
            "claim_cost_median",
            "claim_cost_sum",
            "claim_cost_max",
        ]
        member_claims_agg = agg_df.copy()
    else:
        member_claims_agg = (
            claims_analytical_base.groupby("member_id")
            .size()
            .reset_index(name="claims_count")
        )

print("member_claims_agg:", None if member_claims_agg is None else member_claims_agg.shape)
display(member_claims_agg.head(5) if member_claims_agg is not None else pd.DataFrame())

# %% [markdown]
# ## Merge claims aggregates into member base

# %%
member_risk_base = member_master.copy()

if member_claims_agg is not None:
    member_risk_base = member_risk_base.merge(
        member_claims_agg,
        on="member_id",
        how="left"
    )

# fill members with no claims
for col in ["claims_count", "claim_cost_mean", "claim_cost_median", "claim_cost_sum", "claim_cost_max"]:
    if col in member_risk_base.columns:
        member_risk_base[col] = member_risk_base[col].fillna(0)

print("member_risk_base:", member_risk_base.shape)
display(member_risk_base.head(3))

# %% [markdown]
# ## Candidate risk variables review

# %%
candidate_risk_cols = [c for c in [
    "baseline_risk_score",
    "utilization_propensity",
    "acute_event_propensity",
    "misuse_exposure_propensity",
    "claims_count",
    "claim_cost_sum",
    "claim_cost_mean",
    "chronic_condition_count",
    "premium_monthly",
    "pricing_adequacy_ratio",
] if c in member_risk_base.columns]

member_risk_base[candidate_risk_cols].describe()

# %% [markdown]
# ## Define initial targets
#
# We will create:
# - `high_cost_flag`
# - `high_utilization_flag`
# - `high_risk_composite_flag`
# - `cost_bucket`

# %%
# high cost
if "claim_cost_sum" in member_risk_base.columns:
    cost_p80 = member_risk_base["claim_cost_sum"].quantile(0.80)
    cost_p95 = member_risk_base["claim_cost_sum"].quantile(0.95)

    member_risk_base["high_cost_flag"] = (member_risk_base["claim_cost_sum"] >= cost_p80).astype(int)
    member_risk_base["very_high_cost_flag"] = (member_risk_base["claim_cost_sum"] >= cost_p95).astype(int)
else:
    member_risk_base["high_cost_flag"] = 0
    member_risk_base["very_high_cost_flag"] = 0

# high utilization
if "claims_count" in member_risk_base.columns:
    util_p80 = member_risk_base["claims_count"].quantile(0.80)
    util_p95 = member_risk_base["claims_count"].quantile(0.95)

    member_risk_base["high_utilization_flag"] = (member_risk_base["claims_count"] >= util_p80).astype(int)
    member_risk_base["very_high_utilization_flag"] = (member_risk_base["claims_count"] >= util_p95).astype(int)
else:
    member_risk_base["high_utilization_flag"] = 0
    member_risk_base["very_high_utilization_flag"] = 0

# composite risk
components = []

for col in [
    "high_baseline_risk_flag",
    "high_utilization_propensity_flag",
    "high_chronic_burden_flag",
    "high_misuse_exposure_flag",
    "high_cost_flag",
    "high_utilization_flag",
]:
    if col in member_risk_base.columns:
        components.append(col)

member_risk_base["risk_composite_points"] = member_risk_base[components].sum(axis=1)
member_risk_base["high_risk_composite_flag"] = (member_risk_base["risk_composite_points"] >= 3).astype(int)

# cost buckets
if "claim_cost_sum" in member_risk_base.columns:
    member_risk_base["cost_bucket"] = pd.qcut(
        member_risk_base["claim_cost_sum"].rank(method="first"),
        q=4,
        labels=["low", "medium", "high", "very_high"]
    )
else:
    member_risk_base["cost_bucket"] = "low"

# %%
target_summary = pd.DataFrame({
    "target": [
        "high_cost_flag",
        "very_high_cost_flag",
        "high_utilization_flag",
        "very_high_utilization_flag",
        "high_risk_composite_flag",
    ],
    "positive_rate": [
        member_risk_base["high_cost_flag"].mean(),
        member_risk_base["very_high_cost_flag"].mean(),
        member_risk_base["high_utilization_flag"].mean(),
        member_risk_base["very_high_utilization_flag"].mean(),
        member_risk_base["high_risk_composite_flag"].mean(),
    ]
})

target_summary

# %% [markdown]
# ## Quick profiling of targets

# %%
if "archetype" in member_risk_base.columns:
    display(
        member_risk_base.groupby("archetype")[["high_cost_flag", "high_utilization_flag", "high_risk_composite_flag"]]
        .mean()
        .reset_index()
        .sort_values("high_risk_composite_flag", ascending=False)
    )

if "plan_type" in member_risk_base.columns:
    display(
        member_risk_base.groupby("plan_type")[["high_cost_flag", "high_utilization_flag", "high_risk_composite_flag"]]
        .mean()
        .reset_index()
        .sort_values("high_risk_composite_flag", ascending=False)
    )

# %%
selected_model_cols = [c for c in [
    "member_id",
    "age",
    "sex",
    "region",
    "city_tier",
    "socioeconomic_band",
    "employment_status",
    "dependents_n",
    "smoker_flag",
    "alcohol_risk_flag",
    "physical_activity_level",
    "bmi_group",
    "chronic_condition_flag",
    "chronic_condition_count",
    "recurrent_medication_flag",
    "prior_hospitalization_24m_flag",
    "self_management_adherence",
    "archetype",
    "baseline_risk_score",
    "utilization_propensity",
    "acute_event_propensity",
    "misuse_exposure_propensity",
    "price_sensitivity",
    "coverage_preference",
    "network_preference",
    "preferred_plan_type",
    "plan_type",
    "plan_tier",
    "coverage_scope",
    "provider_network_type",
    "deductible_amount",
    "copay_level",
    "annual_coverage_limit",
    "premium_monthly",
    "premium_annual",
    "pricing_adequacy_ratio",
    "claims_count",
    "claim_cost_mean",
    "claim_cost_sum",
    "claim_cost_max",
    "high_cost_flag",
    "very_high_cost_flag",
    "high_utilization_flag",
    "very_high_utilization_flag",
    "risk_composite_points",
    "high_risk_composite_flag",
    "cost_bucket",
] if c in member_risk_base.columns]

member_risk_model_base = member_risk_base[selected_model_cols].copy()
print("member_risk_model_base:", member_risk_model_base.shape)

# %% [markdown]
# ## Save output

# %%
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_file = PROCESSED_DIR / "member_risk_model_base.csv"
member_risk_model_base.to_csv(output_file, index=False)

print("Saved:", output_file)

# %% [markdown]
# ## Final notes

# %%
notes = [
    "Initial risk targets are now defined.",
    "This is a first pragmatic target design for portfolio modeling.",
    "Next notebook will build the first predictive risk scoring model."
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth
member_master: (4000, 63)
claims_analytical_base: (44144, 107)


,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,sales_channel,broker_id,tenure_days,tenure_months_approx,premium_gap,has_dependents_flag,high_chronic_burden_flag,high_baseline_risk_flag,high_utilization_propensity_flag,high_misuse_exposure_flag
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,Corporate,NaN,1559,51.3,4.547474e-13,0,0,1,1,1
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,Direct,NaN,1003,33.0,4.547474e-13,0,0,0,0,0
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,Direct,NaN,917,30.2,-9.094947e-13,1,0,1,1,0


,claim_id,member_id,policy_id,provider_id,claim_date,claim_year,claim_month,service_type,service_category,diagnosis_group,...,contract_type,base_cost_multiplier,diagnostic_intensity,admission_intensity,average_claim_expected,historical_volume_band,historical_suspicion_flag,provider_quality_proxy,fraud_exposure_score,premium_gap
0,CLM000000350,MBR001287,POL001287,PRV00080,2024-01-01,2024,2024-01,Diagnostics,Laboratory,General,...,Standard,0.892698,Low,Medium,197.96,High,0,High,0.197,4.547474e-13
1,CLM000000547,MBR001823,POL001823,PRV00129,2024-01-01,2024,2024-01,Diagnostics,Diagnostics,Preventive,...,Standard,1.552278,Low,Medium,388.66,Low,0,Low,0.292,0.000000e+00
2,CLM000000911,MBR003002,POL003002,PRV00110,2024-01-01,2024,2024-01,Outpatient,Follow-up Consultation,Minor Acute,...,Preferred,1.044202,Medium,Medium,201.38,High,0,Low,0.227,0.000000e+00


Detected claim cost column: None
member_claims_agg: (3848, 2)


,member_id,claims_count
0,MBR000001,49
1,MBR000002,1
2,MBR000003,24
3,MBR000004,1
4,MBR000005,15


member_risk_base: (4000, 64)


,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,broker_id,tenure_days,tenure_months_approx,premium_gap,has_dependents_flag,high_chronic_burden_flag,high_baseline_risk_flag,high_utilization_propensity_flag,high_misuse_exposure_flag,claims_count
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,NaN,1559,51.3,4.547474e-13,0,0,1,1,1,49.0
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,NaN,1003,33.0,4.547474e-13,0,0,0,0,0,1.0
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,NaN,917,30.2,-9.094947e-13,1,0,1,1,0,24.0


,archetype,high_cost_flag,high_utilization_flag,high_risk_composite_flag
4,Hyper-Utilizer / Misuse Risk,0.0,1.000000,1.000000
0,Chronic Managed,0.0,0.875706,0.923729
1,Family Planner,0.0,0.012517,0.000000
2,Healthy Low User,0.0,0.000000,0.000000
3,High Acute Risk,0.0,0.000000,0.000000
5,Preventive User,0.0,0.000000,0.000000


,plan_type,high_cost_flag,high_utilization_flag,high_risk_composite_flag
4,Managed Review,0.0,1.000000,1.000000
0,Chronic Care,0.0,0.875706,0.923729
1,Essential,0.0,0.000000,0.000000
2,Family Plus,0.0,0.012517,0.000000
3,High Protection,0.0,0.000000,0.000000
5,Standard,0.0,0.000000,0.000000


member_risk_model_base: (4000, 44)
Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\processed\member_risk_model_base.csv
1. Initial risk targets are now defined.
2. This is a first pragmatic target design for portfolio modeling.
3. Next notebook will build the first predictive risk scoring model.
